# React — Styling & Tailwind CSS

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> Styling has to be seen, and it does not belong in the playground. LESSON 11 and 12 are
> done in `react-scratch`, the throwaway project you created in LESSON 2. LESSON 13 comes
> back to the notebook, because building class strings is plain JavaScript.
>
> Everything here is preparation for the first mini-project, which is next.

## LESSON 11 — Three ways to style a component

React has no opinion about CSS. These are the three mechanisms you get out of the box with
Vite, and each one has a job.

### 1. A plain CSS file, imported

```jsx
import "./App.css";

export default function App() {
  return <div className="card">…</div>;
}
```

The import is not a browser feature — Vite sees it and bundles the CSS (LESSON 3 mentioned
this). The rules are **global**: a `.card` written anywhere matches every `.card` on the
page, whichever file it came from.

### 2. CSS Modules — scoped to one component

Name the file `Something.module.css` and import it as an object:

```css
/* Card.module.css */
.card { border: 1px solid #cbd5e1; padding: 1rem; }
```

```jsx
import styles from "./Card.module.css";

export default function Card({ children }) {
  return <div className={styles.card}>{children}</div>;
}
```

`styles.card` is a **generated** class name. Build the project and the CSS actually
contains something like `._card_12hfz_1`, with the JavaScript referring to that same
generated name. Two components can both call their class `.card` and never collide, because
neither of them is really called `card` by the time it reaches the browser.

### 3. An inline style object

From LESSON 9:

```jsx
<div style={{ width: `${percent}%` }} />
```

For values you cannot know in advance. React's own guidance: use `style` for dynamic
values, and classes for everything else.

### Choosing

| situation | use |
|---|---|
| page resets, base typography, one small project | a global CSS file |
| a component's own appearance, in a codebase with many components | CSS Modules |
| a value computed while the app runs — a width, a position, a colour from data | inline `style` |

There is a fourth option — **utility classes**, which is Tailwind. That is the next lesson,
and it is what the mini-project uses.

### Key Notes

- Importing a CSS file is a build-tool feature; the rules it contains are **global**.
- `*.module.css` gives generated, collision-proof class names via `styles.x`.
- Inline `style` is for values computed at runtime, not for general appearance.
- Nothing here is React-specific except how the class name reaches the element.

### Example

**In your project.** Three components, three mechanisms, side by side.

```jsx
// global — matches every .banner on the page
import "./banner.css";
export function Banner() {
  return <div className="banner">Sale</div>;
}
```

```jsx
// module — the generated name cannot collide
import styles from "./Badge.module.css";
export function Badge() {
  return <span className={styles.badge}>New</span>;
}
```

```jsx
// inline — the width is worked out while the app runs
export function Bar() {
  const percent = Math.round((37 / 50) * 100);
  return <div style={{ width: `${percent}%`, background: "seagreen" }} />;
}
```

### Exercise

**In your project** — `react-scratch`, from LESSON 2. Run `npm run dev` there.

The point of this exercise is to see a collision happen and then see it stop happening.

1. Create two components, `Card` and `Panel`, each in its own file, each returning a `<div>`
   with some text.
2. Give each one a **global** CSS file — `card.css` and `panel.css` — and in **both** define
   a class called `.box`: one with a red border, one with a blue border. Use
   `className="box"` in both components and render both in `App`.
3. Look at the page. Both boxes are the same colour. Open DevTools, inspect one, and work
   out which file won and why.
4. Now convert **only** `Panel` to a CSS Module: rename its file to `Panel.module.css`,
   import it as `styles`, and use `className={styles.box}`. The collision should disappear.
5. Run `npm run build`, open the CSS file in `dist/assets/`, and find the generated class
   name. Compare it with the one that is still called `box`.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

Five styling jobs. For each, name the mechanism you would reach for — global CSS, CSS
Module, or inline `style` — and say why in a few words.

1. `box-sizing: border-box` for every element in the app.
2. The border radius and padding of a `Card` component used on six screens.
3. The height of a chart bar, which comes from a number in the data.
4. A `.title` class used by two unrelated components that want to look different.
5. The brand colour, needed in eleven components.

Then one question worth thinking about: number 5 is the awkward one. What goes wrong if you
answer "inline style" — and what goes wrong if you answer "copy the hex code into eleven
CSS Modules"?

## LESSON 12 — Tailwind CSS

A **utility class** does one thing. `p-6` sets padding. `text-sm` sets a font size.
`flex` sets `display: flex`. Instead of writing a rule called `.card` and deciding what it
contains, you compose the appearance out of small classes at the point of use:

```jsx
<div className="p-6 bg-white rounded-lg border border-slate-200">
  <h2 className="text-2xl font-bold text-slate-900">Revenue</h2>
  <p className="mt-2 text-sm text-slate-600">12,400 EUR</p>
</div>
```

**What you gain:** no class names to invent, no second file to keep in sync, and no dead
CSS — you can see exactly what an element looks like without leaving the line.
**What you pay:** the markup gets long. That is the honest trade, and it is why components
matter: you write the long version once, inside `Card`, and use `<Card>` everywhere else.

### Setup with Vite

```bash
npm install tailwindcss @tailwindcss/vite
```

In `vite.config.js`, add the plugin next to the React one:

```js
import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'
import tailwindcss from '@tailwindcss/vite'

export default defineConfig({
  plugins: [react(), tailwindcss()],
})
```

Then replace everything in `src/index.css` with one line:

```css
@import "tailwindcss";
```

That is the whole setup. Tailwind v4 needs no JavaScript config file for anything this
course does.

### The classes you actually need

| job | classes |
|---|---|
| spacing | `p-4` `px-6` `py-2` `m-2` `mt-4` `gap-4` |
| size | `w-full` `max-w-md` `h-10` |
| layout | `flex` `grid` `grid-cols-3` `items-center` `justify-between` |
| text | `text-sm` `text-2xl` `font-bold` `text-slate-600` |
| surface | `bg-white` `border` `border-slate-200` `rounded-lg` `shadow` |

Numbers are steps on a spacing scale, not pixels: the number multiplies a base unit, so
`p-4` is twice the padding of `p-2` — and the steps in between exist too, `p-3` included.
Colours are `name-shade`, from `50` (lightest) to `950`.

### Prefixes: responsive and state

```jsx
<p className="text-sm md:text-base hover:text-slate-900">…</p>
```

A prefix says *when* the utility applies. `md:` means "from the medium breakpoint upwards",
so unprefixed classes are the small-screen default and prefixed ones take over as the
screen grows — build the narrow layout first, then widen it. `hover:` and `focus:` work the
same way, for state rather than size.

### Only what you use ships

Tailwind scans your source and emits CSS for the classes it finds there. Build a project
using `p-6` and never `bg-fuchsia-700`, and the output contains the first and not the
second. This is why the framework can offer thousands of utilities without shipping them.

It is also a constraint, and LESSON 13 is about living with it.

### Key Notes

- A utility class sets one thing; you compose appearance at the point of use.
- Setup is three steps: install, add the Vite plugin, `@import "tailwindcss";`.
- Unprefixed = all sizes. `md:` and friends apply from that width **up**; `hover:`/`focus:`
  apply in that state.
- Only the classes found in your source are generated.

### Example

**In your project.** A responsive KPI card. One column on a phone, three across from the
medium breakpoint up.

```jsx
export default function App() {
  return (
    <div className="p-6 bg-slate-50 min-h-screen">
      <h1 className="text-2xl font-bold text-slate-900">Dashboard</h1>

      <div className="mt-6 grid grid-cols-1 gap-4 md:grid-cols-3">
        <div className="p-4 bg-white rounded-lg border border-slate-200">
          <p className="text-sm text-slate-500">Revenue</p>
          <p className="mt-1 text-xl font-bold text-slate-900">12,400 EUR</p>
        </div>
      </div>
    </div>
  );
}
```

### Exercise

**In your project** — `react-scratch` again.

1. Install Tailwind and wire it up with the three steps above. Confirm it works before
   going further: put `className="text-3xl font-bold text-blue-600"` on something and check
   the browser.
2. Delete the leftover boilerplate from `src/App.css` and `src/index.css` — `index.css`
   should contain the `@import` line and nothing else.
3. Build a small dashboard header: a title on the left, a subtitle underneath it, and a
   `Refresh` button pushed to the right edge. Use `flex`, `items-center` and
   `justify-between`.
4. Under it, place three cards in a row on a wide screen that stack into one column on a
   narrow one. Use `grid`, `grid-cols-1`, `md:grid-cols-3` and `gap-4`.
5. Narrow the browser window until the layout changes. That width is the `md` breakpoint.
6. Give the cards a `hover:` effect — a different background or border.

Keep the markup in `App` for now. Splitting it into components is the mini-project's job.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

Two experiments and a prediction.

1. **Prove the "only what you use" claim.** Run `npm run build`, open the CSS file in
   `dist/assets/`, and search it for a class you used — then for one you did not, such as
   `bg-fuchsia-700`. Report what you find.
2. **Now break it.** In `App.jsx`, write this and look at the page:

   ```jsx
   const shade = 600;
   <p className={`text-blue-${shade}`}>Hello</p>
   ```

   The text is not blue. Before you read on, say why — given what you just proved in step 1,
   and remembering that Tailwind reads your **source file**, not your running app.
3. Search the built CSS for `text-blue-600`. Is it there? What would Tailwind have had to do
   to get this right?

## LESSON 13 — Building class strings in JavaScript

Back to the notebook, because this is plain JavaScript.

A `className` is just a string, so you can build it with code. There is one rule you have
to respect while doing it.

### Class names must appear complete in your source

Tailwind reads your source files as **text**. It never runs them. So this fails:

```jsx
const shade = 600;
<p className={`text-blue-${shade}`}>Hello</p>
```

Tailwind finds `text-blue-` in the file, which is not a class, and generates nothing.
Search the built CSS afterwards and `text-blue-600` is simply not there. The element gets a
class that no rule matches, and the text stays black — no error, no warning.

**The rule: write complete class names and choose between them.**

```js
const colour = isActive ? "text-blue-600" : "text-slate-500";
```

Both strings appear in full, so both get generated, and one of them is picked at runtime.

### Joining pieces

Most elements want a fixed base plus something conditional. A four-line helper handles it:

```js
function cx(...values) {
  return values.filter(Boolean).join(" ");
}

cx("px-4 py-2 rounded-lg", isActive && "bg-blue-600", isWide && "w-full");
```

`filter(Boolean)` throws away `false`, `undefined`, `null` and `""`, which is what lets you
drop conditions straight into the argument list. Without it, a false condition would print
the word `false` into your class attribute.

### Many variants: use a lookup

```js
const toneClasses = {
  ok: "bg-green-100 text-green-800",
  warn: "bg-amber-100 text-amber-800",
  bad: "bg-red-100 text-red-800",
};

toneClasses[tone];
```

Every class name is written out somewhere in the file, so Tailwind sees them all. This
scales better than a chain of ternaries, and it is the pattern you will keep using.

### Where the condition comes from

Today: a local variable or a function parameter. In topic 6 the condition will arrive as a
**prop**, and in topic 9 it will come from **state**. The technique does not change — only
the source of the value does.

### Key Notes

- Tailwind scans source **text**, so a class name must appear complete and unbroken.
- Never build a class name by interpolation. Choose between whole strings instead.
- `cx(...)` with `filter(Boolean).join(" ")` joins a base with conditional extras.
- A lookup object beats stacked ternaries once there are more than two variants.

### Example

**Runnable — plain JS.** Nothing React-specific: a string is produced, and in a component it
would go straight into `className`.

In [ ]:
function l13cx(...values) {
  return values.filter(Boolean).join(" ");
}

const l13toneClasses = {
  ok: "bg-green-100 text-green-800",
  warn: "bg-amber-100 text-amber-800",
  bad: "bg-red-100 text-red-800",
};

function l13badge(tone, isLarge) {
  return l13cx(
    "inline-block rounded-full font-medium",
    l13toneClasses[tone],
    isLarge ? "px-4 py-2 text-base" : "px-2 py-1 text-xs",
  );
}

console.log(l13badge("ok", false));
console.log(l13badge("bad", true));

// filter(Boolean) is what keeps a false condition out of the string
console.log(l13cx("p-2", false, undefined, "", "border"));

### Exercise

Write `l13buttonClasses(variant, isDisabled)` in the cell below. It returns one complete
class string.

- Always included: `"px-4 py-2 rounded-lg font-medium"`.
- The variant adds:
  - `primary` → `"bg-blue-600 text-white"`
  - `secondary` → `"bg-slate-200 text-slate-900"`
  - `danger` → `"bg-red-600 text-white"`
- When `isDisabled` is true, also add `"opacity-50 cursor-not-allowed"`.
- An unknown variant falls back to `secondary` rather than producing `undefined` in the
  string.

Write your own `cx` first — do not reuse the one from the example cell, since writing it is
half the exercise.

Log all four of these:

```js
l13buttonClasses("primary", false)
l13buttonClasses("danger", true)
l13buttonClasses("secondary", false)
l13buttonClasses("ghost", false)
```

Then, in a comment, answer: `` `bg-${variant}-600` `` would have been shorter than the
lookup object. Give both reasons it is wrong — one about Tailwind, one about the three
variants above.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Part 1.** For each snippet, will Tailwind generate the class? Answer yes or no and say
why.

```js
// 1
`p-${size}`

// 2
size === "lg" ? "p-6" : "p-2"

// 3
const classes = "p-6";   // then className={classes}

// 4
const map = { lg: "p-6", sm: "p-2" };   // then className={map[size]}
```

**Part 2.** A colleague cannot get `bg-fuchsia-700` to generate, so they add this line to
the file:

```js
// bg-fuchsia-700 bg-fuchsia-800 bg-fuchsia-900
```

It works — the classes appear in the built CSS. Explain why it works, then give two reasons
you would not ship it. What should they do instead?

In [ ]:
// Your code here